# 03 — Gold : une question métier = une table, exportée dans PostgreSQL

```
silver/vols ─┬─► gold/ponctualite_mensuelle ─┐
silver/      ├─► gold/causes_retard          │
 aeroports   ├─► gold/aeroports              ├──► PostgreSQL (tables gold_*) ──► Streamlit
 compagnies  ├─► gold/heure_jour             │
             └─► gold/routes                 ┘
```

Chaque table répond à une question de la directrice des opérations et suit le **contrat gold** fixé dans le plan
du projet (noms de colonnes en `snake_case`, pourcentages de 0 à 100).

| Table | Une ligne = | Répond à |
|---|---|---|
| `gold_ponctualite_mensuelle` | mois × compagnie | Contexte : tendance et saisonnalité |
| `gold_causes_retard` | mois × compagnie × cause | Constat : quelles causes, contrôlables ou non |
| `gold_aeroports` | aéroport d'origine | Constat : carte des points noirs |
| `gold_heure_jour` | jour de semaine × heure prévue | Constat : heatmap des créneaux à risque |
| `gold_routes` | origine × destination | Recommandation : routes à traiter en premier |

## Définitions communes

- **vols** : vols programmés (y compris annulés et détournés).
- **vols opérés** : ni annulés, ni détournés. Seuls ces vols ont une heure d'arrivée, donc un retard.
- **à l'heure** : vol opéré arrivé avec moins de 15 min de retard (`retard_15 = 0`). Le seuil de 15 min est celui du
  DOT ; le dénominateur, lui, est un choix du projet (voir ci-dessous).
- `pct_a_l_heure = 100 × vols_a_l_heure / vols_operes` ; `pct_annules = 100 × vols_annules / vols` ;
  `retard_moyen_min` = moyenne du retard à l'arrivée des vols opérés (les avances comptent en négatif).

**Écart avec le chiffre officiel** : le DOT compte les vols annulés et détournés comme « pas à l'heure » et les
garde dans le dénominateur ; il publie **78,10 %** de vols à l'heure pour 2024
([BTS, rapport de décembre 2024](https://www.bts.gov/newsroom/air-travel-consumer-report-december-2024-full-year-2024-numbers)).
Ce projet les **exclut** du dénominateur (on mesure la ponctualité des vols qui ont effectivement volé, les
annulations ayant leur propre indicateur), d'où un taux un peu plus élevé. La section 8 calcule les deux. Le petit
écart restant avec le DOT vient de ce qu'il agrège par compagnie qui vend le billet, la BTS par compagnie qui opère le vol.

**Colonnes de comptes** (`vols_operes`, `vols_a_l_heure`, `vols_annules`) : elles s'ajoutent au contrat pour que le
dashboard puisse **ré-agréger correctement** quand on filtre (plusieurs compagnies, plusieurs mois). Faire la moyenne
de pourcentages donnerait le même poids à un mois de 1 000 vols et à un mois de 100 000 vols ; il faut recalculer
`somme(vols_a_l_heure) / somme(vols_operes)`.

## 1. Session Spark et lecture du silver

In [1]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F, Window
from commun.spark_utils import creer_session, format_tables, vers_postgres, LAKE

spark = creer_session("03_gold")
assert format_tables(spark) == "delta", "Delta Lake n'est pas actif : Kernel → Restart."

vols = spark.read.format("delta").load(f"{LAKE}/silver/vols")
aeroports = spark.read.format("delta").load(f"{LAKE}/silver/aeroports")
compagnies = spark.read.format("delta").load(f"{LAKE}/silver/compagnies")

n_silver = vols.count()
print(f"silver/vols : {n_silver:,} vols".replace(",", " "))

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.


Session prête. Spark UI : http://localhost:4040


silver/vols : 7 078 487 vols


Les indicateurs de ponctualité sont les mêmes dans quatre tables : on les écrit une seule fois.

In [2]:
opere = ~F.col("annule") & ~F.col("detourne")


def indicateurs():
    """Agrégats communs : comptes, puis pourcentages et retard moyen calculés à partir des comptes."""
    return [
        F.count("*").alias("vols"),
        F.sum(opere.cast("int")).alias("vols_operes"),
        F.sum((opere & (F.col("retard_15") == 0)).cast("int")).alias("vols_a_l_heure"),
        F.sum(F.col("annule").cast("int")).alias("vols_annules"),
        F.avg(F.when(opere, F.col("retard_arrivee_min"))).alias("_retard_moyen"),
    ]


def pourcentages(df):
    """Ajoute pct_a_l_heure, pct_annules et retard_moyen_min (arrondis pour l'affichage).

    try_divide renvoie NULL au lieu d'une erreur quand le dénominateur vaut 0 (ex. une route dont tous les
    vols ont été annulés : aucun vol opéré, donc pas de taux de ponctualité).
    """
    return (df
            .withColumn("pct_a_l_heure", F.round(100 * F.try_divide("vols_a_l_heure", "vols_operes"), 2))
            .withColumn("pct_annules", F.round(100 * F.try_divide("vols_annules", "vols"), 2))
            .withColumn("retard_moyen_min", F.round("_retard_moyen", 1))
            .drop("_retard_moyen"))


def ecrire(df, nom, table_pg=None):
    """Écrit la table dans le data lake (gold/<nom>) puis dans PostgreSQL (gold_<nom> par défaut)."""
    chemin = f"{LAKE}/gold/{nom}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).save(chemin)
    relue = spark.read.format("delta").load(chemin)
    vers_postgres(relue, table_pg or f"gold_{nom}")
    return relue

## 2. `gold_ponctualite_mensuelle` — contexte

Une ligne par mois et par compagnie. `mois` est une vraie date (le 1ᵉʳ du mois) pour que les graphiques temporels
la placent correctement sur l'axe.

In [3]:
ponctualite = pourcentages(
    vols.groupBy(F.to_date(F.date_trunc("month", "date_vol")).alias("mois"), "compagnie_code")
        .agg(*indicateurs())
).join(compagnies, "compagnie_code", "left").select(
    "mois", "compagnie_code", "compagnie_nom", "vols", "pct_a_l_heure", "pct_annules", "retard_moyen_min",
    "vols_operes", "vols_a_l_heure", "vols_annules",
).orderBy("mois", "compagnie_code")

ponctualite = ecrire(ponctualite, "ponctualite_mensuelle")
ponctualite.show(5)

Table PostgreSQL « gold_ponctualite_mensuelle » écrite.


+----------+--------------+-----------------+-----+-------------+-----------+----------------+-----------+--------------+------------+
|      mois|compagnie_code|    compagnie_nom| vols|pct_a_l_heure|pct_annules|retard_moyen_min|vols_operes|vols_a_l_heure|vols_annules|
+----------+--------------+-----------------+-----+-------------+-----------+----------------+-----------+--------------+------------+
|2024-01-01|            9E|     Endeavor Air|16972|        76.75|       4.16|            10.6|      16219|         12448|         706|
|2024-01-01|            AA|American Airlines|77300|        70.67|       1.61|            18.1|      75843|         53597|        1248|
|2024-01-01|            AS|  Alaska Airlines|17775|        72.44|      17.55|            10.4|      14495|         10500|        3119|
|2024-01-01|            B6|  JetBlue Airways|19579|        70.98|       1.71|            13.4|      19171|         13607|         334|
|2024-01-01|            DL|  Delta Air Lines|74384|    

## 3. `gold_causes_retard` — constat

La BTS répartit le retard de chaque vol arrivé avec 15 min ou plus en cinq causes (en minutes) :

| Cause | Colonne silver | Contrôlable par la compagnie ? |
|---|---|---|
| `compagnie` | `retard_compagnie_min` (équipage, maintenance, bagages…) | oui |
| `avion_retard` | `retard_avion_min` (l'avion arrive en retard du vol précédent) | oui |
| `meteo` | `retard_meteo_min` | non |
| `nas` | `retard_nas_min` (contrôle aérien, trafic, aéroport) | non |
| `securite` | `retard_securite_min` | non |

**Calcul de la part** : `part_pct = 100 × minutes de la cause / minutes de toutes les causes`, sur le mois et la
compagnie. On somme les **minutes**, puis on divise : la version précédente faisait la moyenne des pourcentages
vol par vol, ce qui donnait autant de poids à un retard de 16 min qu'à un retard de 5 h.
Pour ré-agréger dans le dashboard (plusieurs mois ou compagnies), il faut de même sommer `minutes` puis diviser.

In [4]:
CAUSES = [   # (nom de la cause, colonne silver, contrôlable)
    ("compagnie", "retard_compagnie_min", True),
    ("avion_retard", "retard_avion_min", True),
    ("meteo", "retard_meteo_min", False),
    ("nas", "retard_nas_min", False),
    ("securite", "retard_securite_min", False),
]

par_cause = (vols.filter(opere)
             .groupBy(F.to_date(F.date_trunc("month", "date_vol")).alias("mois"), "compagnie_code")
             .agg(*[F.sum(col).alias(nom) for nom, col, _ in CAUSES]))

# Passage en format long : une ligne par cause (stack = « unpivot »)
expression = ", ".join(f"'{nom}', {nom}" for nom, _, _ in CAUSES)
controlables = [nom for nom, _, c in CAUSES if c]

causes = (par_cause
          .select("mois", "compagnie_code", F.expr(f"stack({len(CAUSES)}, {expression}) AS (cause, minutes)"))
          .withColumn("minutes", F.coalesce("minutes", F.lit(0.0)))
          .withColumn("controlable", F.col("cause").isin(controlables))
          .withColumn("part_pct", F.round(
              100 * F.col("minutes") / F.sum("minutes").over(Window.partitionBy("mois", "compagnie_code")), 2))
          .select("mois", "compagnie_code", "cause", "controlable", "minutes", "part_pct")
          .orderBy("mois", "compagnie_code", "cause"))

causes = ecrire(causes, "causes_retard")
causes.show(5)

Table PostgreSQL « gold_causes_retard » écrite.
+----------+--------------+------------+-----------+--------+--------+
|      mois|compagnie_code|       cause|controlable| minutes|part_pct|
+----------+--------------+------------+-----------+--------+--------+
|2024-01-01|            9E|avion_retard|       true|138168.0|   40.63|
|2024-01-01|            9E|   compagnie|       true| 90471.0|    26.6|
|2024-01-01|            9E|       meteo|      false| 46799.0|   13.76|
|2024-01-01|            9E|         nas|      false| 64555.0|   18.98|
|2024-01-01|            9E|    securite|      false|    73.0|    0.02|
+----------+--------------+------------+-----------+--------+--------+
only showing top 5 rows


## 4. `gold_aeroports` — constat (carte)

Une ligne par aéroport d'**origine** : les départs, leur ponctualité et leurs annulations, avec les coordonnées
venues du référentiel silver (OurAirports) pour la carte.

In [5]:
aeroports_gold = pourcentages(
    vols.groupBy(F.col("origine").alias("code_iata")).agg(*indicateurs())
).join(aeroports.select("code_iata", "ville", "etat", "lat", "lon"), "code_iata", "left").select(
    "code_iata", "ville", "etat", "lat", "lon",
    F.col("vols").alias("departs"), "pct_a_l_heure", "retard_moyen_min", "pct_annules",
    "vols_operes", "vols_a_l_heure", "vols_annules",
).orderBy(F.desc("departs"))

aeroports_gold = ecrire(aeroports_gold, "aeroports")
aeroports_gold.show(5)

Table PostgreSQL « gold_aeroports » écrite.
+---------+-----------------+----+----------------+------------------+-------+-------------+----------------+-----------+-----------+--------------+------------+
|code_iata|            ville|etat|             lat|               lon|departs|pct_a_l_heure|retard_moyen_min|pct_annules|vols_operes|vols_a_l_heure|vols_annules|
+---------+-----------------+----+----------------+------------------+-------+-------------+----------------+-----------+-----------+--------------+------------+
|      ATL|          Atlanta|  GA|         33.6367|        -84.428101| 341896|        80.15|             6.8|        1.0|     337648|        270621|        3410|
|      DFW|Dallas/Fort Worth|  TX|       32.896801|        -97.038002| 313574|         72.9|            13.9|       1.86|     306864|        223700|        5837|
|      DEN|           Denver|  CO|       39.860027|       -104.673792| 308639|        77.23|             7.7|       1.13|     304317|        23503

## 5. `gold_heure_jour` — constat (heatmap)

Une ligne par jour de semaine (1 = lundi … 7 = dimanche) et heure de départ **prévue** (0–23, heure locale de
l'aéroport). `pct_retard` = part des vols opérés arrivés avec 15 min de retard ou plus (= 100 − `pct_a_l_heure`).

In [6]:
heure_jour = pourcentages(
    vols.groupBy("jour_semaine", F.col("heure_depart_prevue").alias("heure")).agg(*indicateurs())
).withColumn("pct_retard", F.round(100 - 100 * F.try_divide("vols_a_l_heure", "vols_operes"), 2)).select(
    "jour_semaine", "heure", "vols", "pct_retard", "vols_operes", "vols_a_l_heure", "vols_annules",
).orderBy("jour_semaine", "heure")

heure_jour = ecrire(heure_jour, "heure_jour")
print(f"{heure_jour.count()} créneaux (7 jours × 24 h = 168 au maximum)")
heure_jour.orderBy(F.desc("pct_retard")).show(5)

Table PostgreSQL « gold_heure_jour » écrite.
168 créneaux (7 jours × 24 h = 168 au maximum)


+------------+-----+-----+----------+-----------+--------------+------------+
|jour_semaine|heure| vols|pct_retard|vols_operes|vols_a_l_heure|vols_annules|
+------------+-----+-----+----------+-----------+--------------+------------+
|           7|   20|51168|     33.48|      49986|         33252|        1073|
|           5|   20|50047|     33.19|      49040|         32765|         952|
|           4|   20|48260|     32.98|      47226|         31653|         927|
|           7|   19|59195|      32.9|      57818|         38798|        1257|
|           5|   19|59550|     32.29|      58335|         39499|        1104|
+------------+-----+-----+----------+-----------+--------------+------------+
only showing top 5 rows


## 6. `gold_routes` — recommandation

Une ligne par route (origine → destination). Le **score de priorité** approche le nombre de vols en retard sur
l'année : `score_priorite = vols × (1 − pct_a_l_heure / 100)` (formule du contrat). Il est un peu plus élevé que le
nombre exact, car `vols` compte aussi les vols annulés ; pour le nombre exact de vols en retard, utiliser
`vols_operes − vols_a_l_heure`. Il combine le volume et le taux de retard : une route
très fréquentée et souvent en retard passe devant une petite route au taux pire mais au faible impact.
Le score est calculé avec le pourcentage non arrondi, puis arrondi à l'unité.

Une route sans aucun vol opéré (tous annulés ou détournés) n'a pas de taux de ponctualité (`NULL`) ; son score vaut
**0** (aucun retard mesurable). Sans cela, PostgreSQL placerait ces `NULL` en tête d'un `ORDER BY score_priorite DESC`.

In [7]:
routes = (vols.groupBy("origine", "destination").agg(*indicateurs())
          .withColumn("score_priorite",
                      F.coalesce(F.round(F.col("vols") * (1 - F.try_divide("vols_a_l_heure", "vols_operes"))).cast("int"), F.lit(0))))
routes = pourcentages(routes).select(
    "origine", "destination", "vols", "pct_a_l_heure", "retard_moyen_min", "score_priorite",
    "vols_operes", "vols_a_l_heure", "vols_annules",
).orderBy(F.desc("score_priorite"))

routes = ecrire(routes, "routes")
print(f"{routes.count():,} routes".replace(",", " "))
toutes_annulees = routes.filter(F.col("vols_operes") == 0)
print(f"dont {toutes_annulees.count()} sans aucun vol opéré (tous annulés ou détournés) : "
      "pct_a_l_heure y est vide, score_priorite = 0")
toutes_annulees.show(truncate=False)
routes.show(10)

Table PostgreSQL « gold_routes » écrite.
6 805 routes


dont 13 sans aucun vol opéré (tous annulés ou détournés) : pct_a_l_heure y est vide, score_priorite = 0
+-------+-----------+----+-------------+----------------+--------------+-----------+--------------+------------+
|origine|destination|vols|pct_a_l_heure|retard_moyen_min|score_priorite|vols_operes|vols_a_l_heure|vols_annules|
+-------+-----------+----+-------------+----------------+--------------+-----------+--------------+------------+
|GJT    |ASE        |1   |NULL         |NULL            |0             |0          |0             |0           |
|RAP    |XWA        |1   |NULL         |NULL            |0             |0          |0             |0           |
|EUG    |OTH        |1   |NULL         |NULL            |0             |0          |0             |0           |
|ONT    |PSP        |1   |NULL         |NULL            |0             |0          |0             |1           |
|LAX    |BUR        |1   |NULL         |NULL            |0             |0          |0             |0     

+-------+-----------+-----+-------------+----------------+--------------+-----------+--------------+------------+
|origine|destination| vols|pct_a_l_heure|retard_moyen_min|score_priorite|vols_operes|vols_a_l_heure|vols_annules|
+-------+-----------+-----+-------------+----------------+--------------+-----------+--------------+------------+
|    LAX|        SFO|11526|        67.13|            14.9|          3789|      11390|          7646|         104|
|    SFO|        LAX|11498|        76.07|             8.8|          2751|      11377|          8655|         116|
|    ORD|        LGA|10969|        76.66|            10.4|          2560|      10695|          8199|         226|
|    SAN|        SFO| 6364|        60.75|            21.2|          2498|       6240|          3791|          98|
|    ATL|        MCO| 8935|        72.65|            15.1|          2444|       8771|          6372|         145|
|    LAX|        LAS| 9603|        74.93|            10.5|          2407|       9511|   

## 7. Contrôles de cohérence

Avant de livrer au dashboard, on vérifie que les tables gold redonnent bien le silver :
- chaque table qui découpe tous les vols (par mois, aéroport, créneau, route) totalise **le même nombre de vols** que silver ;
- dans `gold_causes_retard`, les parts d'un mois × compagnie font **100 %** ;
- aucune valeur manquante là où le dashboard en a besoin (coordonnées, noms de compagnies, pourcentages).

In [8]:
totaux = {
    "silver/vols": n_silver,
    "gold_ponctualite_mensuelle": ponctualite.agg(F.sum("vols")).first()[0],
    "gold_aeroports": aeroports_gold.agg(F.sum("departs")).first()[0],
    "gold_heure_jour": heure_jour.agg(F.sum("vols")).first()[0],
    "gold_routes": routes.agg(F.sum("vols")).first()[0],
}
for nom, n in totaux.items():
    print(f"{nom:<28} {n:>10,} vols".replace(",", " "))
assert len(set(totaux.values())) == 1, "Une table gold ne totalise pas le même nombre de vols que silver"

parts = causes.groupBy("mois", "compagnie_code").agg(F.sum("part_pct").alias("total"))
ecart = parts.agg(F.max(F.abs(F.col("total") - 100))).first()[0]
print(f"Parts des causes : écart maximal à 100 % = {ecart:.2f} point (arrondis)")
assert ecart < 0.1

manquants = {
    "compagnie sans nom": ponctualite.filter(F.col("compagnie_nom").isNull()).count(),
    "aéroport sans coordonnées": aeroports_gold.filter(F.col("lat").isNull()).count(),
    "créneau sans heure": heure_jour.filter(F.col("heure").isNull()).count(),
    "route sans score": routes.filter(F.col("score_priorite").isNull()).count(),
    "route sans pourcentage (hors routes 100 % annulées)":
        routes.filter(F.col("pct_a_l_heure").isNull() & (F.col("vols_operes") > 0)).count(),
}
print(manquants)
assert sum(manquants.values()) == 0

silver/vols                   7 078 487 vols
gold_ponctualite_mensuelle    7 078 487 vols
gold_aeroports                7 078 487 vols
gold_heure_jour               7 078 487 vols
gold_routes                   7 078 487 vols
Parts des causes : écart maximal à 100 % = 0.02 point (arrondis)


{'compagnie sans nom': 0, 'aéroport sans coordonnées': 0, 'créneau sans heure': 0, 'route sans score': 0, 'route sans pourcentage (hors routes 100 % annulées)': 0}


## 8. Relecture depuis PostgreSQL

Le dashboard lira PostgreSQL, pas le data lake. On relit donc les tables avec `lire()` (pandas, sur des tables
gold de petite taille), comme le fera Streamlit.

Pour les parcourir à la souris : http://localhost:8080 (Adminer — système *PostgreSQL*, serveur `postgres`,
`cours` / `cours`, base `vols`).

In [9]:
from commun.donnees import lire

import pandas as pd

tables = lire("SELECT table_name FROM information_schema.tables "
              "WHERE table_schema = 'public' AND left(table_name, 5) = 'gold_' ORDER BY table_name")["table_name"]
# (pas de LIKE 'gold_%' : avec psycopg2, un « % » dans le SQL est lu comme un paramètre et lire() échoue)
pd.DataFrame({"table": tables,
              "lignes": [lire(f"SELECT count(*) AS n FROM {t}")["n"][0] for t in tables]})

,table,lignes
0,gold_aeroports,348
1,gold_causes_retard,900
2,gold_heure_jour,168
3,gold_ponctualite_mensuelle,180
4,gold_routes,6805


In [10]:
# Ponctualité nationale 2024, ré-agrégée correctement à partir des comptes.
# pct_a_l_heure_dot : même calcul avec la méthode du DOT (annulés et détournés au dénominateur).
lire("""
    SELECT SUM(vols)                                                  AS vols,
           ROUND(100.0 * SUM(vols_a_l_heure) / SUM(vols_operes), 1)    AS pct_a_l_heure,
           ROUND(100.0 * SUM(vols_a_l_heure) / SUM(vols), 1)           AS pct_a_l_heure_dot,
           ROUND(100.0 * SUM(vols_annules) / SUM(vols), 2)             AS pct_annules,
           ROUND((SUM(retard_moyen_min * vols_operes) / SUM(vols_operes))::numeric, 1) AS retard_moyen_min
    FROM gold_ponctualite_mensuelle
""")

,vols,pct_a_l_heure,pct_a_l_heure_dot,pct_annules,retard_moyen_min
0,7078487.0,79.2,77.9,1.36,7.0


In [11]:
# Part des minutes de retard par cause sur l'année (somme des minutes, puis division)
lire("""
    SELECT cause, controlable,
           ROUND((SUM(minutes) / 60)::numeric)                               AS heures,
           ROUND((100 * SUM(minutes) / SUM(SUM(minutes)) OVER ())::numeric, 1) AS part_pct
    FROM gold_causes_retard
    GROUP BY cause, controlable
    ORDER BY part_pct DESC
""")

,cause,controlable,heures,part_pct
0,avion_retard,True,695211.0,40.6
1,compagnie,True,585643.0,34.2
2,nas,False,326781.0,19.1
3,meteo,False,102617.0,6.0
4,securite,False,2997.0,0.2


In [12]:
# Les 10 routes à traiter en premier
lire("SELECT * FROM gold_routes ORDER BY score_priorite DESC LIMIT 10")

,origine,destination,vols,pct_a_l_heure,retard_moyen_min,score_priorite,vols_operes,vols_a_l_heure,vols_annules
0,LAX,SFO,11526,67.13,14.9,3789,11390,7646,104
1,SFO,LAX,11498,76.07,8.8,2751,11377,8655,116
2,ORD,LGA,10969,76.66,10.4,2560,10695,8199,226
3,SAN,SFO,6364,60.75,21.2,2498,6240,3791,98
4,ATL,MCO,8935,72.65,15.1,2444,8771,6372,145
5,LAX,LAS,9603,74.93,10.5,2407,9511,7127,87
6,MCO,ATL,8967,74.03,14.3,2329,8821,6530,137
7,SEA,SFO,6599,66.42,15.2,2216,6512,4325,86
8,FLL,ATL,7702,71.27,15.9,2213,7592,5411,94
9,LAS,LAX,9559,77.06,7.3,2193,9481,7306,74


## 9. Échantillon pour le graphe sémantique (`04_semantique`)

Le graphe RDF de `04_semantique` (rdflib) ne peut pas contenir 7 millions de vols : il décrit **tous les aéroports
et toutes les compagnies**, les **50 routes prioritaires** et un **échantillon de 500 vols** sur ces routes. Ces tables
sont préparées ici et copiées dans PostgreSQL sous le préfixe `sem_`, pour être lues avec `lire()`, sans session Spark :
le notebook sémantique n'a besoin que de pandas et de rdflib.

| Table PostgreSQL | Contenu | Lignes |
|---|---|---|
| `sem_aeroports` | référentiel silver + départs, part des départs nationaux, `est_hub` | 348 |
| `sem_compagnies` | code et nom | 15 |
| `sem_routes` | les 50 routes au plus fort `score_priorite` | 50 |
| `sem_vols` | 10 vols par route prioritaire (500), avec retards, annulation, déroutement et minutes par cause | 500 |

**Définition d'un hub** (classe `Hub` de l'ontologie) : un aéroport qui assure **au moins 1 % des départs nationaux**.
C'est le seuil des *large hubs* de la FAA, défini sur les passagers embarqués
([14 CFR § 398.2](https://www.law.cornell.edu/cfr/text/14/398.2)) ; faute de données passagers, on l'applique aux vols :
c'est une approximation.

**Échantillon reproductible** : sur chaque route, on garde les 10 vols dont la clé a le plus petit hachage
(`xxhash64`). Contrairement à `rand()`, dont le résultat dépend du découpage des données en partitions, ce tirage
donne toujours les mêmes 500 vols : le graphe ne change pas d'une exécution à l'autre.

In [13]:
TOP_ROUTES = 50        # routes prioritaires décrites dans le graphe
VOLS_PAR_ROUTE = 10    # vols échantillonnés par route
SEUIL_HUB_PCT = 1.0    # part minimale des départs nationaux pour être un hub

# Aéroports : référentiel silver + volume de départs + statut de hub
total_departs = aeroports_gold.agg(F.sum("departs")).first()[0]
sem_aeroports = (aeroports
    .join(aeroports_gold.select("code_iata", "departs"), "code_iata", "left")
    .withColumn("departs", F.coalesce("departs", F.lit(0)))
    .withColumn("part_departs_pct", F.round(100 * F.col("departs") / F.lit(total_departs), 3))
    .withColumn("est_hub", F.col("part_departs_pct") >= SEUIL_HUB_PCT)
    .orderBy(F.desc("departs")))
sem_aeroports = ecrire(sem_aeroports, "sem_aeroports", "sem_aeroports")

sem_compagnies = ecrire(compagnies.orderBy("compagnie_code"), "sem_compagnies", "sem_compagnies")

# Routes prioritaires (origine et destination départagent les ex aequo : classement stable)
sem_routes = ecrire(routes.orderBy(F.desc("score_priorite"), "origine", "destination").limit(TOP_ROUTES),
                    "sem_routes", "sem_routes")

hubs = sem_aeroports.filter("est_hub")
print(f"{hubs.count()} hubs (≥ {SEUIL_HUB_PCT} % des départs) :",
      ", ".join(r["code_iata"] for r in hubs.collect()))

Table PostgreSQL « sem_aeroports » écrite.


Table PostgreSQL « sem_compagnies » écrite.


Table PostgreSQL « sem_routes » écrite.


30 hubs (≥ 1.0 % des départs) : ATL, DFW, DEN, ORD, CLT, LAX, PHX, LAS, SEA, LGA, MCO, BOS, DCA, SFO, DTW, EWR, MSP, JFK, IAH, SLC, MIA, PHL, BNA, BWI, SAN, FLL, AUS, MDW, TPA, DAL


In [14]:
# Vols : 10 par route prioritaire, choisis par hachage de leur identifiant
id_vol = F.concat_ws("_", F.date_format("date_vol", "yyyyMMdd"), "compagnie_code", "numero_vol",
                     "origine", F.lpad(F.col("heure_depart_prevue").cast("string"), 2, "0"))

rang = F.row_number().over(Window.partitionBy("origine", "destination")
                                 .orderBy(F.xxhash64("id_vol"), "id_vol"))

sem_vols = (vols
    .join(F.broadcast(sem_routes.select("origine", "destination")), ["origine", "destination"])
    .withColumn("id_vol", id_vol)
    .withColumn("_rang", rang)
    .filter(F.col("_rang") <= VOLS_PAR_ROUTE)
    .select("id_vol", "date_vol", "compagnie_code", "numero_vol", "origine", "destination",
            "heure_depart_prevue", "distance_miles", "retard_depart_min", "retard_arrivee_min", "retard_15",
            "annule", "code_annulation", "detourne", *[col for _, col, _ in CAUSES])
    .orderBy("origine", "destination", "id_vol"))
sem_vols = ecrire(sem_vols, "sem_vols", "sem_vols")

n = sem_vols.count()
assert n == TOP_ROUTES * VOLS_PAR_ROUTE, n
assert sem_vols.select("id_vol").distinct().count() == n, "id_vol doit être unique"

sem_vols.select(
    F.count("*").alias("vols"),
    F.sum((~F.col("annule") & ~F.col("detourne") & (F.col("retard_15") == 1)).cast("int")).alias("en_retard"),
    F.sum(F.col("annule").cast("int")).alias("annules"),
    F.sum(F.col("detourne").cast("int")).alias("detournes"),
    F.countDistinct("compagnie_code").alias("compagnies"),
    F.countDistinct("origine").alias("aeroports_origine"),
).show()

Table PostgreSQL « sem_vols » écrite.


+----+---------+-------+---------+----------+-----------------+
|vols|en_retard|annules|detournes|compagnies|aeroports_origine|
+----+---------+-------+---------+----------+-----------------+
| 500|      120|      9|        0|        12|               19|
+----+---------+-------+---------+----------+-----------------+



In [15]:
# Ce que lira 04_semantique, sans Spark
lire("SELECT * FROM sem_vols ORDER BY id_vol LIMIT 5")

,id_vol,date_vol,compagnie_code,numero_vol,origine,destination,heure_depart_prevue,distance_miles,retard_depart_min,retard_arrivee_min,retard_15,annule,code_annulation,detourne,retard_compagnie_min,retard_avion_min,retard_meteo_min,retard_nas_min,retard_securite_min
0,20240101_B6_623_JFK_14,2024-01-01,B6,623,JFK,LAX,14,2475.0,-4.0,-41.0,0,False,None,False,None,None,None,None,None
1,20240101_OO_3779_SEA_09,2024-01-01,OO,3779,SEA,SFO,9,679.0,-4.0,-17.0,0,False,None,False,None,None,None,None,None
2,20240101_UA_1399_DEN_07,2024-01-01,UA,1399,DEN,SEA,7,1024.0,-3.0,-25.0,0,False,None,False,None,None,None,None,None
3,20240102_UA_1597_LAX_22,2024-01-02,UA,1597,LAX,SFO,22,337.0,18.0,7.0,0,False,None,False,None,None,None,None,None
4,20240103_DL_1036_MCO_06,2024-01-03,DL,1036,MCO,ATL,6,404.0,-4.0,-12.0,0,False,None,False,None,None,None,None,None


## C'est réussi quand…

- la console RustFS montre les dossiers `gold/…` et Adminer les 5 tables `gold_*` et les 4 tables `sem_*` ;
- les contrôles de la section 7 passent (même nombre de vols partout, parts des causes à 100 %) ;
- la ponctualité nationale relue depuis PostgreSQL est cohérente avec le silver (79,2 %, soit environ 78 % avec la
  méthode du DOT, proche des 78,10 % publiés).

Suite : `04_semantique` construit le graphe à partir des tables `sem_*` ; `05_exploration` et le dashboard lisent les tables `gold_*`.

In [16]:
spark.stop()